# 01 — Exploratory Analysis: Google Search Trends

**Dataset:** Real Google Trends data via pytrends (live API)  
**Topics:** 14 keywords across Tech, Health, Finance  
**Timeframe:** Last 5 years (weekly granularity)  
**Author:** Sierra Napier | e3-ai.com


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.signal import find_peaks
import warnings
warnings.filterwarnings('ignore')

df_ww = pd.read_csv('data/interest_over_time_worldwide.csv', index_col=0, parse_dates=True)
df_us = pd.read_csv('data/interest_over_time_us.csv', index_col=0, parse_dates=True)
df_region = pd.read_csv('data/interest_by_region_us.csv')

print(f'Worldwide time series: {df_ww.shape}')
print(f'US time series: {df_us.shape}')
print(f'US regional: {df_region.shape}')
print(f'Keywords: {list(df_ww.columns)}')

Worldwide time series: (262, 14)
US time series: (262, 14)
US regional: (714, 6)
Keywords: ['AI', 'machine learning', 'ChatGPT', 'Bitcoin', 'Tesla', 'Netflix', 'Amazon', 'telehealth', 'mental health', 'fitness', 'inflation', 'recession', 'stock market', 'crypto']


## 1.1 Overall Trend Overview — Worldwide

In [2]:
fig = make_subplots(rows=3, cols=1, subplot_titles=('Tech', 'Health', 'Finance'), vertical_spacing=0.08)

tech = ['AI', 'machine learning', 'ChatGPT', 'Bitcoin', 'Tesla', 'Netflix', 'Amazon']
health = ['telehealth', 'mental health', 'fitness']
finance = ['inflation', 'recession', 'stock market', 'crypto']

for kw in tech:
    fig.add_trace(go.Scatter(x=df_ww.index, y=df_ww[kw], name=kw, mode='lines'), row=1, col=1)
for kw in health:
    fig.add_trace(go.Scatter(x=df_ww.index, y=df_ww[kw], name=kw, mode='lines'), row=2, col=1)
for kw in finance:
    fig.add_trace(go.Scatter(x=df_ww.index, y=df_ww[kw], name=kw, mode='lines'), row=3, col=1)

fig.update_layout(height=900, title_text='Worldwide Search Interest (2021–2026)', showlegend=False)
fig.show()

## 1.2 Peak Detection — When did each topic spike?

In [3]:
peaks = {}
for col in df_ww.columns:
    p, props = find_peaks(df_ww[col], height=50, distance=10)
    if len(p) > 0:
        peaks[col] = df_ww.index[p].strftime('%Y-%m-%d').tolist()
    else:
        p2, _ = find_peaks(df_ww[col], height=20, distance=10)
        peaks[col] = df_ww.index[p2].strftime('%Y-%m-%d').tolist() if len(p2) > 0 else []

peak_df = pd.DataFrame([(k, ', '.join(v[:3])) for k, v in peaks.items() if v], columns=['Keyword', 'Top Peaks (dates)'])
print(peak_df.to_string(index=False))

     Keyword                  Top Peaks (dates)
          AI 2025-03-30, 2025-06-29, 2025-09-14
     ChatGPT 2025-03-30, 2025-06-08, 2025-09-14
     Bitcoin                         2021-05-16
     Netflix 2021-07-04, 2021-10-03, 2022-01-02
      Amazon 2021-06-20, 2021-10-03, 2021-12-12
   inflation 2022-05-08, 2022-08-07, 2025-08-10
stock market             2024-08-04, 2025-04-06
      crypto 2021-10-31, 2022-01-16, 2022-05-08


## 1.3 Correlation Matrix — Which topics move together?

In [4]:
corr = df_ww.corr()
fig = px.imshow(corr, text_auto='.2f', aspect='auto', color_continuous_scale='RdBu_r',
                title='Topic Correlation Matrix (Worldwide)', zmin=-1, zmax=1)
fig.update_layout(height=700)
fig.show()

## 1.4 Growth Rate — YoY change by category

In [5]:
# Calculate YoY growth using 52-week windows
growth = {}
for col in df_ww.columns:
    recent = df_ww[col].iloc[-52:].mean()
    prior = df_ww[col].iloc[-104:-52].mean()
    if prior > 0:
        growth[col] = round((recent - prior) / prior * 100, 1)
    else:
        growth[col] = None

growth_df = pd.DataFrame(list(growth.items()), columns=['Keyword', 'YoY_Growth_%']).sort_values('YoY_Growth_%', ascending=False)
fig = px.bar(growth_df, x='Keyword', y='YoY_Growth_%', color='YoY_Growth_%',
             color_continuous_scale='RdYlGn', title='YoY Growth Rate (%) — 52-week avg')
fig.show()

## 1.5 Regional Interest Map — US States

In [6]:
pivot_region = df_region.pivot(index='geoName', columns='keyword', values='interest').fillna(0)
fig = px.choropleth(df_region, locations='geoCode', color='interest',
                    hover_name='geoName', scope='usa',
                    animation_frame='keyword',
                    color_continuous_scale='Plasma',
                    title='Search Interest by US State (animated by keyword)')
fig.update_layout(height=500)
fig.show()

## 1.6 Category Summaries

In [7]:
summary = df_ww.describe().T[['mean', 'std', 'min', 'max']].round(2)
summary['category'] = summary.index.map(lambda x: 'Tech' if x in tech else ('Health' if x in health else 'Finance'))
print(summary.sort_values('mean', ascending=False))

                   mean    std   min    max category
Amazon            74.99   8.66  59.0  100.0     Tech
crypto            34.40  16.30  15.0  100.0  Finance
Netflix           31.37   3.32  23.0   44.0     Tech
AI                28.06  23.55   4.0  100.0     Tech
ChatGPT           25.88  26.28   0.0   82.0     Tech
stock market      19.36   9.33  11.0   93.0  Finance
inflation         14.77   5.99   6.0   37.0  Finance
fitness            7.54   1.29   6.0   14.0   Health
Bitcoin            6.91   2.70   4.0   24.0     Tech
Tesla              6.03   1.28   4.0   14.0     Tech
recession          3.08   2.30   1.0   17.0  Finance
mental health      2.24   0.74   1.0    5.0   Health
machine learning   0.89   0.51   0.0    3.0     Tech
telehealth         0.04   0.19   0.0    1.0   Health
